# 44 — Análise qualitativa dos erros da classe `mercado`

Orquestrador focado **exclusivamente** nos erros que envolvem a classe `mercado` — a classe central da dissertação (formulação binária `mercado` vs `outros`; e a classe `mercado` no esquema multiclasse 7+other). Toda lógica reusável vive em `src/economy_classifier/error_analysis.py`; este notebook só configura, agrega, amostra e exporta.

## Perguntas-âncora

1. **Quais notícias `mercado` foram classificadas erradas?** No binário, são os **FNs** (`y_true=mercado, y_pred=outros`). No multiclasse, são os **outflows** (`y_true=mercado, y_pred=X`).
2. **Para quais classes os `mercado` errados foram atribuídos no multiclasse?** Mapa `mercado → poder/colunas/cotidiano/...` por modelo.
3. **Os erros multiclasse são os mesmos do binário?** Cruzamento por `index` na **mesma família**: o multiclasse erra exatamente onde o binário também erra, ou um pega o que o outro deixa?

## Perguntas analíticas adicionais

| # | Pergunta | Output |
|---|---|---|
| 4 | Que editorias se confundem **com** `mercado`? (FP binário; `X→mercado` multiclasse) | `binary/<run>/mercado_fp_by_category.csv`, `multiclass/<run>/mercado_inflow_by_true.csv` |
| 5 | Subcategorias de `mercado` mais erradas? (macro vs empresas vs financeiro) | `binary/<run>/mercado_fn_by_subcategory.csv` |
| 6 | Drift temporal: erros de `mercado` se concentram em algum ano? | `binary/<run>/mercado_fn_by_date.csv` |
| 7 | "Leads curtos": FNs são textos curtos? | `binary/<run>/mercado_fn_by_text_length.csv` |
| 8 | "Perto do limiar" vs "alta confiança em `outros`"? (apenas TF-IDF/BERT — LLM tem score determinístico) | `binary/<run>/mercado_fn_by_confidence.csv` |
| 9 | Quais notícias `mercado` **todos** os modelos erram? (hard set) | `disagreement/hard_mercado_binary.csv`, `hard_mercado_multiclass.csv` |
| 10 | Concordância entre famílias **só nas mercado**? | `disagreement/mercado_only_binary.csv`, `mercado_only_multiclass.csv` |

## Pipeline

```
predictions.csv + test.parquet
    -> load_predictions_with_text()
    -> build_binary_error_pool()        # FN = mercado errado; FP = "confundido com mercado"
    -> build_multiclass_error_pool(focus_classes={"mercado"})  # outflow + inflow
    -> cross_binary_multiclass_errors_for_class()   # Pergunta 3
    -> filter_disagreement_by_true_class() / hard_examples_for_class()  # Perguntas 9-10
    -> summarize_errors_by_*() -> stratified_error_sample(seed=2026)
    -> export_annotation_template()  -> [anotação manual]
    -> load_annotated_sample() + summarize_annotations()  -> adjusted_correctness
```

## Schema de anotação (vocabulário controlado)

| valor | quando aplicar |
|---|---|
| `rotulagem_editorial` | Texto é tematicamente `mercado`, mas a editoria colocou em outra seção. **Erro do rótulo, não do modelo.** |
| `tema_misto` | Texto cruza fronteiras (ex.: política econômica). Tanto `mercado` quanto a outra classe são defensáveis. |
| `modelo_erra` | Texto é claramente `mercado`; o modelo perdeu sinais textuais óbvios. **Erro do modelo.** |
| `ambiguo` | Texto é curto/inespecífico — humano também não classifica com confiança. |

Colunas auxiliares: `subtema_real`, `editorialmente_economia` (bool), `signal_palavras`, `notas`.

## Tamanhos amostrais (foco assimétrico em FN)

- **Binário por modelo**: 50 FN + 30 FP (FN é o foco — `mercado` que escapou). 80 anotações × N modelos.
- **Multiclasse por modelo**: 10 por par `mercado→X` (outflow) + 10 por par `X→mercado` (inflow), pares pequenos contribuem todos.
- **Hard mercado set**: anotar **100%** (universo pequeno; cada item é candidato a re-rotulagem).
- **Disagreement-mercado**: 30 anotações (10 por padrão), cross-família.

**Reproducibilidade**: todo sampling usa `DEFAULT_SEED=2026`. Rodar duas vezes produz CSVs idênticos.


## 0. Bootstrap (Colab + local)

Mesma lógica do notebook 41.

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "train.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), (
            f"Falta colab_splits.zip em {DRIVE_BASE}. "
            "Rode scripts/colab_pack.py local e suba o zip."
        )
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")
        print("Splits extraidos em", SPLITS_DIR)

    RUNS_DIR = DRIVE_BASE / "runs"
    OUT_DIR = DRIVE_BASE / "error_analysis"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_DIR = REPO_DIR / "artifacts" / "runs"
    OUT_DIR = REPO_DIR / "artifacts" / "error_analysis"

OUT_DIR.mkdir(parents=True, exist_ok=True)
assert RUNS_DIR.exists(), f"RUNS_DIR não encontrado: {RUNS_DIR}"
assert SPLITS_DIR.exists(), f"SPLITS_DIR não encontrado: {SPLITS_DIR}"
print("RUNS_DIR:", RUNS_DIR)
print("SPLITS_DIR:", SPLITS_DIR)
print("OUT_DIR:", OUT_DIR)

## 1. Configuração

Runs binárias e multiclasse por **família** (TF-IDF / BERT / LLM). Para responder à Pergunta 3 (cruzamento binário ↔ multiclasse), cada família precisa de um par binário+multiclasse alinhados — `FAMILY_PAIRS` faz esse mapeamento. Runs ausentes (sem `predictions.csv` no disco) são ignoradas com `[skip]`.

Tamanhos amostrais assimétricos: 50 FN + 30 FP no binário (FN é o foco), 10 por par no multiclasse, 10 por padrão no disagreement-mercado, 100% do hard set.

In [ ]:
import json
import pandas as pd

from economy_classifier.error_analysis import (
    DEFAULT_SEED,
    build_binary_error_pool,
    build_disagreement_pool,
    build_multiclass_error_pool,
    cross_binary_multiclass_errors_for_class,
    detect_task,
    export_annotation_template,
    filter_disagreement_by_true_class,
    hard_examples_for_class,
    load_predictions_with_text,
    stratified_error_sample,
    summarize_errors_by_category,
    summarize_errors_by_confidence,
    summarize_errors_by_date,
    summarize_errors_by_text_length,
)

# Runs binárias por família (one representante por família).
BINARY_RUNS = {
    "tfidf_logreg": "tfidf_logreg_binary_test_set",
    "tfidf_nb": "tfidf_nb_binary_test_set",
    "bert_bertimbau": "bert_bertimbau_binary_test_set",
    "llm_qwen_zero_shot": "llm_qwen2_5_7b_instruct_binary_zero_shot_test_set",
}

# Runs multiclasse por família — alinhadas com as binárias quando possível.
MULTICLASS_RUNS = {
    "tfidf_nb": "tfidf_nb_multiclass_test_set",
    "bert_bertimbau": "bert_bertimbau_multiclass_test_set",
    "llm_qwen_zero_shot": "llm_qwen2_5_7b_instruct_multiclass_zero_shot_test_set",
}

# Pares binário↔multiclasse da MESMA família (Pergunta 3 — cruzamento).
FAMILY_PAIRS = {
    "tfidf_nb": ("tfidf_nb", "tfidf_nb"),
    "bert_bertimbau": ("bert_bertimbau", "bert_bertimbau"),
    "llm_qwen_zero_shot": ("llm_qwen_zero_shot", "llm_qwen_zero_shot"),
}

# Foco: tudo gira em torno de `mercado`.
TARGET_CLASS = "mercado"

# Tamanhos de amostra (assimétrico — FN é o foco).
N_BINARY_SAMPLE = {"FN": 50, "FP": 30}
N_PER_PAIR_MULTICLASS = 10
N_PER_PATTERN_DISAGREEMENT = 10

# Função utilitária: filtra dict de runs àquelas com predictions.csv presente.
def _filter_existing(run_map: dict[str, str]) -> dict[str, str]:
    out = {}
    for label, run_name in run_map.items():
        path = RUNS_DIR / run_name / "predictions.csv"
        if path.exists():
            out[label] = run_name
        else:
            print(f"[skip] {label}: {path.relative_to(RUNS_DIR.parent) if RUNS_DIR in path.parents else path} ausente")
    return out

# Função utilitária: lê coverage do result_card.json (LLMs podem ter <1.0).
def _coverage_warning(run_name: str) -> None:
    card_path = RUNS_DIR / run_name / "result_card.json"
    if not card_path.exists():
        return
    try:
        card = json.loads(card_path.read_text())
        cov = card.get("metrics", {}).get("coverage")
        if cov is not None and cov < 1.0:
            n_eval = card.get("metrics", {}).get("n_eval_samples", "?")
            print(f"  [warn] {run_name}: coverage={cov:.2%} (n_eval={n_eval}) — comparações cross-família devem alinhar índices")
    except (json.JSONDecodeError, KeyError):
        pass

print("DEFAULT_SEED:", DEFAULT_SEED)
print("TARGET_CLASS:", TARGET_CLASS)
print("\nbinary runs disponíveis:")
BINARY_RUNS = _filter_existing(BINARY_RUNS)
for label in BINARY_RUNS:
    _coverage_warning(BINARY_RUNS[label])
print("  ", list(BINARY_RUNS))
print("\nmulticlass runs disponíveis:")
MULTICLASS_RUNS = _filter_existing(MULTICLASS_RUNS)
for label in MULTICLASS_RUNS:
    _coverage_warning(MULTICLASS_RUNS[label])
print("  ", list(MULTICLASS_RUNS))
print("\npares binário↔multiclasse válidos:")
FAMILY_PAIRS = {
    fam: (b, m)
    for fam, (b, m) in FAMILY_PAIRS.items()
    if b in BINARY_RUNS and m in MULTICLASS_RUNS
}
print("  ", list(FAMILY_PAIRS))


## 2. Carregar o test set + perfil da classe `mercado`

O test set fixo (10% dos dados, `seed=2026`) é a base para todos os pools. Antes de qualquer análise, perfilamos `mercado`: quantas notícias são, e como se distribuem por subcategoria editorial. Isso ancora o sentido de "X% dos `mercado` foram errados" na seção 3.

In [ ]:
test_df = pd.read_parquet(SPLITS_DIR / "test.parquet")
print("test shape:", test_df.shape)
print("colunas:", list(test_df.columns))

print("\nlabel binário (1=mercado):")
print(test_df['label'].value_counts().to_dict())

print("\nlabel multiclasse:")
print(test_df['label_multi'].value_counts().to_dict())

mercado_mask = test_df['label_multi'] == TARGET_CLASS
n_mercado = int(mercado_mask.sum())
print(f"\n=== Perfil de `mercado` no test set ===")
print(f"n_mercado = {n_mercado} ({n_mercado/len(test_df):.1%} do test set)")

mercado_subset = test_df[mercado_mask]
if 'subcategory' in mercado_subset.columns:
    print("\nsubcategorias de `mercado` (top 15):")
    sub_counts = mercado_subset['subcategory'].fillna('<missing>').value_counts().head(15)
    print(sub_counts.to_string())

if 'date' in mercado_subset.columns:
    print("\ndistribuição temporal de `mercado` (por ano):")
    years = pd.to_datetime(mercado_subset['date'], errors='coerce').dt.year
    print(years.value_counts().sort_index().to_string())


## 3. Erros binários da classe `mercado`

Para cada run binária:

- **FN pool** = `y_true=mercado, y_pred=outros` → as notícias `mercado` que escaparam (Pergunta 1).
- **FP pool** = `y_true=outros, y_pred=mercado` → as notícias que foram **confundidas com** `mercado` (Pergunta 4).

Salvamos agregados específicos do foco (subcategoria/categoria/data/comprimento/confiança), além das tabelas genéricas. **Nota sobre LLM**: scores são determinísticos (`y_score ∈ {0.0, 1.0}`), então `mercado_fn_by_confidence.csv` colapsa a 2 bins — é avisado em log mas o CSV ainda é gravado para uniformidade.


In [ ]:
binary_pools = {}
binary_fn_pools = {}
binary_fp_pools = {}

for label, run_name in BINARY_RUNS.items():
    preds_path = RUNS_DIR / run_name / "predictions.csv"
    joined = load_predictions_with_text(preds_path, test_df)
    if detect_task(joined) != "binary":
        print(f"[skip] {label}: predictions não são binárias")
        continue

    pool = build_binary_error_pool(joined)
    binary_pools[label] = pool
    fn_pool = pool[pool["error_type"] == "FN"].reset_index(drop=True)
    fp_pool = pool[pool["error_type"] == "FP"].reset_index(drop=True)
    binary_fn_pools[label] = fn_pool
    binary_fp_pools[label] = fp_pool

    out_dir = OUT_DIR / "binary" / label
    out_dir.mkdir(parents=True, exist_ok=True)

    # Tabelas genéricas (mantidas para compatibilidade com 41).
    summarize_errors_by_category(pool, column="error_type").to_csv(
        out_dir / "by_error_type.csv", index=False,
    )

    # Foco em `mercado`: FN específico (= notícias `mercado` que erraram).
    fn_pool.to_csv(out_dir / "mercado_fn.csv", index=False)
    if not fn_pool.empty:
        summarize_errors_by_category(fn_pool, column="subcategory").to_csv(
            out_dir / "mercado_fn_by_subcategory.csv", index=False,
        )
        summarize_errors_by_text_length(fn_pool, n_bins=5).to_csv(
            out_dir / "mercado_fn_by_text_length.csv", index=False,
        )
        summarize_errors_by_date(fn_pool, freq="YS").to_csv(
            out_dir / "mercado_fn_by_date.csv", index=False,
        )
        # Confidence: detect deterministic LLM scores e avisar.
        if "y_score" in fn_pool.columns:
            unique_scores = fn_pool["y_score"].dropna().nunique()
            if unique_scores <= 2:
                print(f"  [warn] {label}: y_score determinístico ({unique_scores} valores únicos) — bins de confiança não são informativos")
            summarize_errors_by_confidence(fn_pool, n_bins=5).to_csv(
                out_dir / "mercado_fn_by_confidence.csv", index=False,
            )

    # FPs: o que se confunde com `mercado`.
    fp_pool.to_csv(out_dir / "mercado_fp.csv", index=False)
    if not fp_pool.empty:
        summarize_errors_by_category(fp_pool, column="category").to_csv(
            out_dir / "mercado_fp_by_category.csv", index=False,
        )
        summarize_errors_by_category(fp_pool, column="subcategory").to_csv(
            out_dir / "mercado_fp_by_subcategory.csv", index=False,
        )

    print(f"\n=== {label}: {len(pool)} erros ({len(fn_pool)} FN + {len(fp_pool)} FP) ===")
    if not fn_pool.empty:
        print("Top subcategorias de `mercado` que escaparam (FN):")
        print(
            summarize_errors_by_category(fn_pool, column="subcategory")
            .head(8).to_string(index=False)
        )
    if not fp_pool.empty:
        print("\nTop categorias confundidas com `mercado` (FP):")
        print(
            summarize_errors_by_category(fp_pool, column="category")
            .head(8).to_string(index=False)
        )


## 4. Erros multiclasse envolvendo `mercado`

`build_multiclass_error_pool(focus_classes={"mercado"})` retorna todos os erros em que `mercado` aparece como `y_true` **ou** `y_pred`. Decompomos em duas categorias:

- **outflow** (`mercado→X`): notícias **realmente** de `mercado` que foram para outra classe → responde Pergunta 2.
- **inflow** (`X→mercado`): outras classes que invadiram `mercado` → responde Pergunta 4 (multiclasse).

Cada CSV traz a tabela ranqueada por destino/origem mais frequente.

In [ ]:
multi_pools = {}
multi_outflow_pools = {}
multi_inflow_pools = {}

for label, run_name in MULTICLASS_RUNS.items():
    preds_path = RUNS_DIR / run_name / "predictions.csv"
    joined = load_predictions_with_text(preds_path, test_df)
    if detect_task(joined) != "multiclass":
        print(f"[skip] {label}: predictions não são multiclasse")
        continue

    pool = build_multiclass_error_pool(joined, focus_classes={TARGET_CLASS})
    multi_pools[label] = pool

    # Outflow: y_true == mercado, y_pred != mercado (mercado errado).
    outflow = pool[pool["y_true"].astype(str) == TARGET_CLASS].reset_index(drop=True)
    # Inflow: y_pred == mercado, y_true != mercado (confundido com mercado).
    inflow = pool[pool["y_pred"].astype(str) == TARGET_CLASS].reset_index(drop=True)
    multi_outflow_pools[label] = outflow
    multi_inflow_pools[label] = inflow

    out_dir = OUT_DIR / "multiclass" / label
    out_dir.mkdir(parents=True, exist_ok=True)

    outflow.to_csv(out_dir / "mercado_outflow.csv", index=False)
    inflow.to_csv(out_dir / "mercado_inflow.csv", index=False)

    if not outflow.empty:
        # mercado→X: agrupa por classe destino (y_pred).
        outflow_by_pred = (
            outflow.groupby(outflow["y_pred"].astype(str))
            .size().reset_index(name="n_errors")
            .rename(columns={0: "y_pred"})
            .sort_values("n_errors", ascending=False)
        )
        outflow_by_pred.columns = ["y_pred", "n_errors"]
        total = outflow_by_pred["n_errors"].sum()
        outflow_by_pred["share"] = (outflow_by_pred["n_errors"] / total).round(4)
        outflow_by_pred.to_csv(out_dir / "mercado_outflow_by_pred.csv", index=False)
        # E por subcategoria de mercado (qual subárea escapou mais).
        summarize_errors_by_category(outflow, column="subcategory").to_csv(
            out_dir / "mercado_outflow_by_subcategory.csv", index=False,
        )

    if not inflow.empty:
        inflow_by_true = (
            inflow.groupby(inflow["y_true"].astype(str))
            .size().reset_index(name="n_errors")
        )
        inflow_by_true.columns = ["y_true", "n_errors"]
        total = inflow_by_true["n_errors"].sum()
        inflow_by_true["share"] = (inflow_by_true["n_errors"] / total).round(4)
        inflow_by_true.to_csv(out_dir / "mercado_inflow_by_true.csv", index=False)

    print(f"\n=== {label}: {len(pool)} erros envolvendo `mercado` ===")
    print(f"  outflow (mercado→X): {len(outflow)}")
    print(f"  inflow  (X→mercado): {len(inflow)}")
    if not outflow.empty:
        print("\n  Top destinos do outflow:")
        print("  " + outflow_by_pred.head(8).to_string(index=False).replace("\n", "\n  "))
    if not inflow.empty:
        print("\n  Top origens do inflow:")
        print("  " + inflow_by_true.head(8).to_string(index=False).replace("\n", "\n  "))


## 5. Cruzamento binário ↔ multiclasse na mesma família (Pergunta 3)

Para cada família com **ambos** os pipelines disponíveis (definida em `FAMILY_PAIRS`), `cross_binary_multiclass_errors_for_class` filtra a notícias que são genuinamente `mercado` (`y_true_binary==1` e `y_true_multi=="mercado"`) e classifica cada uma em quatro padrões:

- `both_correct` — binário e multiclasse acertam.
- `both_wrong` — ambos erram (núcleo duro de `mercado` mal capturado).
- `binary_only_correct` — binário acerta, multiclasse perde.
- `multi_only_correct` — multiclasse acerta, binário perde.

A interseção `both_wrong` é o conjunto onde a família **inteira** falha em ver `mercado` — os candidatos mais fortes a anotação manual.

In [ ]:
cross_pools = {}

for family, (bin_label, multi_label) in FAMILY_PAIRS.items():
    bin_path = RUNS_DIR / BINARY_RUNS[bin_label] / "predictions.csv"
    multi_path = RUNS_DIR / MULTICLASS_RUNS[multi_label] / "predictions.csv"

    bin_joined = load_predictions_with_text(bin_path, test_df)
    multi_joined = load_predictions_with_text(multi_path, test_df)

    cross = cross_binary_multiclass_errors_for_class(
        bin_joined, multi_joined,
        target_class=TARGET_CLASS, binary_positive_label=1,
    )
    cross_pools[family] = cross

    out_dir = OUT_DIR / "cross" / family
    out_dir.mkdir(parents=True, exist_ok=True)
    cross.to_csv(out_dir / "mercado_cross_pool.csv", index=False)

    pattern_counts = cross["agreement_pattern"].value_counts().to_dict()
    n_total = len(cross)
    summary = {
        "family": family,
        "binary_run": BINARY_RUNS[bin_label],
        "multiclass_run": MULTICLASS_RUNS[multi_label],
        "n_mercado_aligned": n_total,
        "n_dropped_binary_only": int(cross.attrs.get("n_dropped_binary_only", 0)),
        "n_dropped_multi_only": int(cross.attrs.get("n_dropped_multi_only", 0)),
        "patterns": {p: int(n) for p, n in pattern_counts.items()},
        "patterns_share": {
            p: round(int(n) / n_total, 4) if n_total else 0.0
            for p, n in pattern_counts.items()
        },
        # κ-like: P(both wrong | at least one wrong) — proxy de "quanto os erros coincidem".
        "share_both_wrong_among_errors": (
            round(
                pattern_counts.get("both_wrong", 0)
                / max(1, n_total - pattern_counts.get("both_correct", 0)),
                4,
            )
            if n_total - pattern_counts.get("both_correct", 0) else 0.0
        ),
    }
    (out_dir / "mercado_cross_summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False)
    )

    print(f"\n=== {family}: {n_total} notícias `mercado` alinhadas ===")
    for pattern, n in sorted(pattern_counts.items(), key=lambda x: -x[1]):
        share = n / n_total if n_total else 0.0
        print(f"  {pattern:>22s}: n={n:>4d}  share={share:.2%}")
    print(f"  share_both_wrong_among_errors: {summary['share_both_wrong_among_errors']:.2%}")
    if summary["n_dropped_binary_only"] or summary["n_dropped_multi_only"]:
        print(f"  [info] índices dropados: binary_only={summary['n_dropped_binary_only']}, multi_only={summary['n_dropped_multi_only']}")


## 6. Disagreement entre famílias **restrito a `mercado`** + hard set

Quando TF-IDF, BERT e LLM dão respostas diferentes para a mesma notícia, quem captura `mercado`? Construímos disagreement pools (binário e multiclasse) e filtramos a `y_true == mercado` / `y_true == 1`. A função `hard_examples_for_class` extrai o caso extremo: **todas** as famílias erram → universo de candidatos a re-rotulagem.

Para uma comparação cross-família **honesta**, restringimos todos os modelos ao mesmo conjunto de índices (interseção). Cards com `coverage < 1.0` reduzem essa interseção; o número final é registrado em `intersection_metadata.json`.

In [ ]:
out_dir = OUT_DIR / "disagreement"
out_dir.mkdir(parents=True, exist_ok=True)

# ---- Binário ----
binary_predictions_by_method = {}
for label, run_name in BINARY_RUNS.items():
    preds = pd.read_csv(RUNS_DIR / run_name / "predictions.csv")
    binary_predictions_by_method[label] = preds

if len(binary_predictions_by_method) >= 2:
    # Restringir todos os modelos à interseção de índices (LLM coverage<1.0).
    idx_sets = [set(p["index"]) for p in binary_predictions_by_method.values()]
    common_idx = set.intersection(*idx_sets) if idx_sets else set()
    print(f"[binary] interseção de índices: {len(common_idx)}")

    aligned = {
        m: p[p["index"].isin(common_idx)].reset_index(drop=True)
        for m, p in binary_predictions_by_method.items()
    }

    disagreement_bin = build_disagreement_pool(aligned, test_df)
    disagreement_bin.to_csv(out_dir / "full_pool_binary.csv", index=False)

    mercado_only_bin = filter_disagreement_by_true_class(disagreement_bin, target_class=1)
    mercado_only_bin.to_csv(out_dir / "mercado_only_binary.csv", index=False)

    hard_bin = hard_examples_for_class(aligned, test_df, target_class=1)
    hard_bin.to_csv(out_dir / "hard_mercado_binary.csv", index=False)

    print(f"  disagreement total: {len(disagreement_bin)}")
    print(f"  mercado_only (y_true=mercado): {len(mercado_only_bin)}")
    if not mercado_only_bin.empty:
        print("  padrão dentro de mercado:")
        print("    " + mercado_only_bin["disagreement_pattern"].value_counts().to_string().replace("\n", "\n    "))
    print(f"  hard mercado (todos erram): {len(hard_bin)}")

    (out_dir / "intersection_metadata_binary.json").write_text(json.dumps({
        "task": "binary",
        "models": list(aligned),
        "n_intersection": len(common_idx),
        "n_per_model_original": {m: len(p) for m, p in binary_predictions_by_method.items()},
    }, indent=2))
else:
    print("[skip-binary] disagreement precisa >=2 famílias com predictions disponíveis")
    disagreement_bin = pd.DataFrame()
    mercado_only_bin = pd.DataFrame()
    hard_bin = pd.DataFrame()

# ---- Multiclasse ----
multi_predictions_by_method = {}
for label, run_name in MULTICLASS_RUNS.items():
    preds = pd.read_csv(RUNS_DIR / run_name / "predictions.csv")
    multi_predictions_by_method[label] = preds

if len(multi_predictions_by_method) >= 2:
    idx_sets = [set(p["index"]) for p in multi_predictions_by_method.values()]
    common_idx = set.intersection(*idx_sets) if idx_sets else set()
    print(f"\n[multiclass] interseção de índices: {len(common_idx)}")

    aligned = {
        m: p[p["index"].isin(common_idx)].reset_index(drop=True)
        for m, p in multi_predictions_by_method.items()
    }

    disagreement_mc = build_disagreement_pool(aligned, test_df)
    disagreement_mc.to_csv(out_dir / "full_pool_multiclass.csv", index=False)

    mercado_only_mc = filter_disagreement_by_true_class(disagreement_mc, target_class=TARGET_CLASS)
    mercado_only_mc.to_csv(out_dir / "mercado_only_multiclass.csv", index=False)

    hard_mc = hard_examples_for_class(aligned, test_df, target_class=TARGET_CLASS)
    hard_mc.to_csv(out_dir / "hard_mercado_multiclass.csv", index=False)

    print(f"  disagreement total: {len(disagreement_mc)}")
    print(f"  mercado_only (y_true=mercado): {len(mercado_only_mc)}")
    if not mercado_only_mc.empty:
        print("  padrão dentro de mercado:")
        print("    " + mercado_only_mc["disagreement_pattern"].value_counts().to_string().replace("\n", "\n    "))
    print(f"  hard mercado (todos erram): {len(hard_mc)}")

    (out_dir / "intersection_metadata_multiclass.json").write_text(json.dumps({
        "task": "multiclass",
        "models": list(aligned),
        "n_intersection": len(common_idx),
        "n_per_model_original": {m: len(p) for m, p in multi_predictions_by_method.items()},
    }, indent=2))
else:
    print("[skip-multiclass] disagreement precisa >=2 famílias")
    disagreement_mc = pd.DataFrame()
    mercado_only_mc = pd.DataFrame()
    hard_mc = pd.DataFrame()


## 7. Amostragem estratificada e templates de anotação

Com os pools formados, geramos os CSVs editáveis para anotação manual:

- **Binário**: 50 FN + 30 FP por modelo (`stratified_error_sample` aceita `dict`).
- **Multiclasse**: 10 por par `mercado→X` (outflow) e 10 por par `X→mercado` (inflow). Pares menores contribuem todos.
- **Hard mercado**: anotar **100%** (universo pequeno e crítico).
- **Disagreement-mercado**: 10 por padrão (`all_wrong`, `majority_wrong_one_right`, `majority_right_one_wrong`, `split`).

Tudo com `seed=2026` — rerodar produz os mesmos arquivos.

In [ ]:
templates_created = []

# ---- Binário (assimétrico FN-foco) ----
for label, pool in binary_pools.items():
    if pool.empty:
        continue
    sample = stratified_error_sample(
        pool,
        n_per_stratum=N_BINARY_SAMPLE,
        stratify_by="error_type",
        seed=DEFAULT_SEED,
    )
    template_path = OUT_DIR / "binary" / label / "annotation_template.csv"
    export_annotation_template(sample, template_path)
    templates_created.append(template_path)
    counts = sample["error_type"].value_counts().to_dict()
    print(f"binary/{label}: {len(sample)} amostras (FN={counts.get('FN', 0)}, FP={counts.get('FP', 0)})")

# ---- Multiclasse: outflow + inflow estratificado por par ----
for label in MULTICLASS_RUNS:
    out_dir = OUT_DIR / "multiclass" / label

    outflow = multi_outflow_pools.get(label)
    if outflow is not None and not outflow.empty:
        sample = stratified_error_sample(
            outflow, n_per_stratum=N_PER_PAIR_MULTICLASS,
            stratify_by="error_type", seed=DEFAULT_SEED,
        )
        path = out_dir / "annotation_template_outflow.csv"
        export_annotation_template(sample, path)
        templates_created.append(path)
        print(f"multiclass/{label} outflow: {len(sample)} amostras")

    inflow = multi_inflow_pools.get(label)
    if inflow is not None and not inflow.empty:
        sample = stratified_error_sample(
            inflow, n_per_stratum=N_PER_PAIR_MULTICLASS,
            stratify_by="error_type", seed=DEFAULT_SEED,
        )
        path = out_dir / "annotation_template_inflow.csv"
        export_annotation_template(sample, path)
        templates_created.append(path)
        print(f"multiclass/{label} inflow:  {len(sample)} amostras")

# ---- Hard mercado set: anotar 100% ----
disagreement_dir = OUT_DIR / "disagreement"
for hard_pool, suffix in [(hard_bin, "binary"), (hard_mc, "multiclass")]:
    if hard_pool is None or hard_pool.empty:
        continue
    pool = hard_pool.copy()
    pool["method"] = f"hard_mercado_{suffix}"
    pool["error_type"] = pool["disagreement_pattern"]
    path = disagreement_dir / f"annotation_template_hard_mercado_{suffix}.csv"
    export_annotation_template(pool, path)
    templates_created.append(path)
    print(f"disagreement/hard_mercado_{suffix}: {len(pool)} amostras (100% do universo)")

# ---- Disagreement-mercado (10 por padrão) ----
for mercado_only_pool, suffix in [(mercado_only_bin, "binary"), (mercado_only_mc, "multiclass")]:
    if mercado_only_pool is None or mercado_only_pool.empty:
        continue
    pool = mercado_only_pool.copy()
    pool["method"] = f"disagreement_mercado_{suffix}"
    pool["error_type"] = pool["disagreement_pattern"]
    sample = stratified_error_sample(
        pool, n_per_stratum=N_PER_PATTERN_DISAGREEMENT,
        stratify_by="disagreement_pattern", seed=DEFAULT_SEED,
    )
    path = disagreement_dir / f"annotation_template_mercado_only_{suffix}.csv"
    export_annotation_template(sample, path)
    templates_created.append(path)
    print(f"disagreement/mercado_only_{suffix}: {len(sample)} amostras")

print(f"\n=== {len(templates_created)} templates de anotação gerados ===")


## 8. Guia de anotação (foco em `mercado`)

Abrir cada `annotation_template.csv` em LibreOffice/Excel/Sheets e preencher. **Sempre se pergunte primeiro: este texto é mesmo `mercado`?** Se sim, **que sinal o modelo perdeu?** (Pergunta 1).

1. **`subtema_real`** — em uma frase, o tema linguístico real. Ex.: "decisão do Copom sobre Selic", "balanço da Vale", "greve em frigorífico exportador", "crítica de filme".
2. **`tipo_erro_anotado`** — exatamente um valor:
   - `rotulagem_editorial` — texto fala de economia/empresas/finanças mas a editoria colocou em outra seção (ex.: BNDES no caderno SP). **Erro do rótulo, não do modelo.**
   - `tema_misto` — texto cruza fronteiras (ex.: "reforma tributária aprovada pelo Congresso" é simultaneamente `poder` e `mercado`). Ambos são defensáveis.
   - `modelo_erra` — texto é claramente `mercado` (ou claramente não-mercado, no FP); o modelo errou sem ter justificativa textual. **Erro do modelo.**
   - `ambiguo` — texto curto/inespecífico demais; humano também não classifica com confiança.
3. **`editorialmente_economia`** — `True` se o texto fala de economia/mercado/empresas/finanças tematicamente, **independente da editoria**. Usado para isolar a fração do corpus que é "economia mal-rotulada".
4. **`signal_palavras`** — 3-6 termos que sinalizam o tema. Ex.: "Selic", "Copom", "BNDES", "lucro líquido", "balanço", "Ibovespa". Útil para discutir features que TF-IDF capta vs. as que ele perde.
5. **`notas`** — observação livre (ex.: "texto truncado, abrir link", "contradiz o título", "lead genérico").

**Critério de qualidade**: dois anotadores independentes devem concordar ≥80% em `tipo_erro_anotado`. Para a dissertação um anotador basta — reportar a limitação.

## 9. Releitura e agregação pós-anotação

Roda depois que os CSVs forem anotados. Valida o vocabulário controlado, agrega por tipo de erro, e produz **`adjusted_correctness`** — fração de erros nominais que **não** são culpa do modelo (= `rotulagem_editorial + tema_misto + ambiguo`). É o número que entra na seção de validade de construto da dissertação.

Edite `ANNOTATED_FILES` apontando para os CSVs já anotados (sufixo `_DONE` ou outro, à sua escolha).

In [ ]:
from economy_classifier.error_analysis import (
    load_annotated_sample,
    summarize_annotations,
)

# Adicione paths para CSVs já anotados. Vazio = pula a agregação.
ANNOTATED_FILES: dict[str, Path] = {
    # "tfidf_nb_binary": OUT_DIR / "binary" / "tfidf_nb" / "annotation_template_DONE.csv",
    # "bert_bertimbau_binary": OUT_DIR / "binary" / "bert_bertimbau" / "annotation_template_DONE.csv",
    # "bert_bertimbau_multiclass_outflow": OUT_DIR / "multiclass" / "bert_bertimbau" / "annotation_template_outflow_DONE.csv",
    # "hard_mercado_binary": OUT_DIR / "disagreement" / "annotation_template_hard_mercado_binary_DONE.csv",
}

summaries_post: dict[str, dict] = {}
for label, path in ANNOTATED_FILES.items():
    if not path.exists():
        print(f"[skip] {label}: {path} não existe")
        continue
    annotated = load_annotated_sample(path, require_complete=True)
    summary = summarize_annotations(annotated)
    summaries_post[label] = summary
    print(f"\n=== {label} ===")
    print(f"  n={summary['n_annotated']}  adjusted_correctness={summary['adjusted_correctness']}")
    print("  tipos:")
    for t, info in summary["by_type"].items():
        print(f"    {t:>22s}: n={info['n']:>3d}  share={info['share']:.2%}")

if summaries_post:
    out = OUT_DIR / "annotation_summaries.json"
    out.write_text(json.dumps(summaries_post, indent=2, ensure_ascii=False))
    print(f"\nresumos salvos em: {out}")


## 10. Resumo da execução — árvore de outputs

Saídas em `OUT_DIR` (= `artifacts/error_analysis/` localmente, `My Drive/economy-classifier/error_analysis/` no Colab):

```
binary/
  <run_label>/
    by_error_type.csv                   # FP/FN counts (compatibilidade)
    mercado_fn.csv                      # FN pool (Pergunta 1)
    mercado_fn_by_subcategory.csv       # subcategorias mais erradas (Q5)
    mercado_fn_by_text_length.csv       # leads curtos? (Q7)
    mercado_fn_by_date.csv              # drift temporal (Q6)
    mercado_fn_by_confidence.csv        # perto-do-limiar vs alta confiança (Q8)
    mercado_fp.csv                      # confusões com `mercado` (Q4)
    mercado_fp_by_category.csv
    mercado_fp_by_subcategory.csv
    annotation_template.csv             # 50 FN + 30 FP para anotar
multiclass/
  <run_label>/
    mercado_outflow.csv                 # mercado → X (Pergunta 2)
    mercado_outflow_by_pred.csv         # ranking de destinos
    mercado_outflow_by_subcategory.csv
    mercado_inflow.csv                  # X → mercado (Q4 multiclasse)
    mercado_inflow_by_true.csv          # ranking de origens
    annotation_template_outflow.csv     # ~10 por par, anotar
    annotation_template_inflow.csv
cross/
  <family>/
    mercado_cross_pool.csv              # binary↔multi alinhado (Pergunta 3)
    mercado_cross_summary.json          # contagem dos 4 padrões + share
disagreement/
  full_pool_binary.csv
  full_pool_multiclass.csv
  mercado_only_binary.csv               # disagreement só nas mercado (Q10)
  mercado_only_multiclass.csv
  hard_mercado_binary.csv               # todos erram (Q9)
  hard_mercado_multiclass.csv
  intersection_metadata_binary.json
  intersection_metadata_multiclass.json
  annotation_template_hard_mercado_binary.csv      # 100% — anotar todos
  annotation_template_hard_mercado_multiclass.csv
  annotation_template_mercado_only_binary.csv     # 30 (10 por padrão)
  annotation_template_mercado_only_multiclass.csv
annotation_summaries.json               # após anotação manual (seção 9)
```

Os CSVs `mercado_*.csv` já são reportáveis na dissertação **antes** de anotar — eles respondem quantitativamente a 8 das 10 perguntas. As anotações manuais alimentam `adjusted_correctness` (validade de construto) e a discussão qualitativa de "por que o modelo perdeu sinais textuais.